<a href="https://colab.research.google.com/github/winicius87/Grammar/blob/main/NBA_2026_Shared_Prediction_with_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_basketball_predictor(num_features):
    # Parallel Inputs for Home and Away rolling statistics
    home_input = layers.Input(shape=(num_features,), name="home_stats")
    away_input = layers.Input(shape=(num_features,), name="away_stats")
    situational_input = layers.Input(shape=(2,), name="situational_context") # e.g., Rest days

    # Shared dense layer to extract latent team strengths
    shared_dense = layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))

    home_features = shared_dense(home_input)
    away_features = shared_dense(away_input)

    # Core Logic: Calculate the structural difference between the teams
    strengths_diff = layers.Subtract()([home_features, away_features])

    # Combine matchup differences with situational features (like home court/rest)
    combined = layers.concatenate([strengths_diff, situational_input])

    # Deep Neural Network Classification Layers
    x = layers.Dense(32, activation='relu')(combined)
    x = layers.Dropout(0.3)(x) # Prevents overfitting to historical noise
    x = layers.Dense(16, activation='relu')(x)

    # Output: Probability of a Home Team Win
    output = layers.Dense(1, activation='sigmoid', name="win_probability")(x)

    model = models.Model(inputs=[home_input, away_input, situational_input], outputs=output)
    return model

model = build_basketball_predictor(num_features=12)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train using EarlyStopping to halt when validation loss stops improving
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    [X_train_home, X_train_away, X_train_situational], y_train,
    validation_data=([X_val_home, X_val_away, X_val_situational], y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)